In [ ]:
import pickle
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class SearchEngine:
    def __init__(self, index_file: str = "index.pkl"):
        self.index_file = Path(index_file)

        self.documents = []
        self.vectorizer = None
        self.tfidf_matrix = None
        self.inverted_index = {}

    def build(self, documents: list[dict], text_field: str):
        """
        Membangun index dari list dokumen.

        Args:
            documents: List data.
            text_field: Field yang akan digunakan untuk pencarian.
        """
        self.documents = documents

        corpus = [doc[text_field] for doc in documents]

        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)

        # Build inverted index
        self.inverted_index = {}

        for doc_id, text in enumerate(corpus):
            for token in text.lower().split():
                self.inverted_index.setdefault(token, set()).add(doc_id)

        self.save()

    def save(self):
        with open(self.index_file, "wb") as f:
            pickle.dump(
                {
                    "documents": self.documents,
                    "vectorizer": self.vectorizer,
                    "tfidf_matrix": self.tfidf_matrix,
                    "inverted_index": self.inverted_index,
                },
                f,
            )

    def load(self):
        with open(self.index_file, "rb") as f:
            data = pickle.load(f)

        self.documents = data["documents"]
        self.vectorizer = data["vectorizer"]
        self.tfidf_matrix = data["tfidf_matrix"]
        self.inverted_index = data["inverted_index"]

    def search(self, keyword: str, top_k: int = 20):
        """
        Melakukan pencarian menggunakan
        Inverted Index + Cosine Similarity.
        """

        # kandidat dokumen
        candidates = set()

        for token in keyword.lower().split():
            candidates |= self.inverted_index.get(token, set())

        if not candidates:
            return []

        query_vector = self.vectorizer.transform([keyword])

        candidate_ids = list(candidates)

        similarities = cosine_similarity(
            query_vector,
            self.tfidf_matrix[candidate_ids],
        )[0]

        results = sorted(
            zip(candidate_ids, similarities),
            key=lambda x: x[1],
            reverse=True,
        )

        return [
            {
                "score": float(score),
                "document": self.documents[idx],
            }
            for idx, score in results[:top_k]
        ]

In [ ]:
"""
penggunaan

documents = [
    {"id": 1, "judul": "Surat Izin Sekolah"},
    {"id": 2, "judul": "Surat Tugas Dosen"},
    {"id": 3, "judul": "Izin Penelitian Mahasiswa"},
]

engine = SearchEngine()

engine.build(
    documents=documents,
    text_field="judul",
)
"""

load db

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv()

from myapp import create_app

app = create_app()
app.app_context().push()

from myapp.config import BASE_DIR

INDEX_DIR = BASE_DIR / "media" / "index"

repo

In [2]:
from myapp.features.surat import repository

services

In [ ]:
from myapp.features.mesin_pencarian import services

use cases

In [4]:
from myapp.features.mesin_pencarian import usecases

buil search engine

In [5]:
repo = repository.SuratMasukRepository()

# search_engine_service = services.SearchEngine(
#     index_file=INDEX_DIR / "index_surat_masuk.pkl"
# )

search_engine_uc = usecases.SearchEngineUseCase(
    repo=repo,
    # search_engine=search_engine_service
)


# build index
# search_engine_uc.build(
#     text_field="isi_singkat",
# )

test pencarian

In [7]:
uc = usecases.SearchEngineUseCase(
    repo=repo,
    tipe="surat_masuk"
)
uc.search("surat", surat_masuk=True)

[{'score': 0.3957514850066099,
  'document': {'id': 11,
   'isi_singkat': 'Surat pengantar berita daerah, informasi, produk hukum',
   'nomor_surat': '481/5049',
   'tanggal_surat': datetime.date(2026, 3, 8),
   'lokasi_arsip': '',
   'nomor_urut': 11,
   'kode_surat': None}}]

In [ ]:
uc = usecases.SearchEngineUseCase()
uc.search("keterangan")

In [ ]:
# test koneksi ke database
from sqlalchemy import text
from myapp.extensions import db
from sqlalchemy import select

result = db.session.execute(text("SELECT 1"))

print(result.scalar())

In [ ]:
from myapp.features.mesin_pencarian import services
from myapp.core.mappers import document_mapper_service

from myapp.config import BASE_DIR

INDEX_DIR = BASE_DIR / "media" / "index"

search_engine = services.SearchEngine(
    index_file = INDEX_DIR / "index.pkl"
)

In [ ]:
from myapp.features.surat import repository
from myapp.features.surat import models

surat_keluar_repo = repository.SuratKeluarRepository()
surat_masuk_repo = repository.SuratMasukRepository()

In [ ]:
documents = [
    {
        "id": surat.id,
        "isi_singkat": surat.isi_singkat,
        "nomor_surat": surat.nomor_surat,
        "tanggal_surat": surat.tanggal_surat,
        "lokasi_arsip": surat.lokasi_arsip,
        "nomor_urut": surat.nomor_urut,
        "kode_surat": surat.kode_surat
    }
    for surat in surat_keluar_repo.get_all()
]
documents

In [ ]:
documents = (
    document_mapper_service
    .DocumentMapperService.db_to_documents(surat_keluar_repo.get_all())
)
documents

In [ ]:
from myapp.config import BASE_DIR
INDEX_DIR = BASE_DIR / "media" / "index"
INDEX_DIR
INDEX_DIR.exists()

In [ ]:
search_engine.index_file

In [ ]:
search_engine.build(
    documents=documents,
    text_field="isi_singkat",
)

In [ ]:
# documents = [
#     {"id": 1, "judul": "Surat Izin Sekolah"},
#     {"id": 2, "judul": "Surat Tugas Dosen"},
#     {"id": 3, "judul": "Izin Penelitian Mahasiswa"},
# ]

engine = SearchEngine()

engine.build(
    documents=documents,
    text_field="isi_singkat",
    base_dir=INDEX_DIR / "index.pkl"
)

In [ ]:
BASE_DIR = Path().resolve().parent / "myapp"
BASE_DIR

In [ ]:


search_engine.build(
    documents=documents,
    text_field="isi_singkat",
)

load app

In [ ]:
# engine = services.SearchEngine(
#     index_file=INDEX_DIR / "index.pkl"
# )

engine = search_engine

engine.load()

In [ ]:
results = engine.search("wilayah")

hasil = [
    {
        "score": item["score"],
        "document": item["document"]
    }
    for item in results
]



In [ ]:
hasil

In [ ]:
db_documents = [item["document"] for item in hasil]

In [ ]:
db_documents

In [ ]:
hasil_dict

In [ ]:
hasil_db = (
    document_mapper_service.DocumentMapperService.documents_to_db(
        db_documents
    )
)
temp = hasil_db[0]

In [ ]:
temp